# Transfer Tax Reform Impact on Housing Production

**Research Question:** How would eliminating San Francisco's real estate transfer tax affect housing production?

**Methodology:** We estimate market values for all SF residential parcels using a gradient boosting model trained on recent sales data. We then calculate how removing the transfer tax would reduce effective construction costs (land is ~2x construction costs, so the tax has a 2x impact on effective costs). Finally, we run the City Economist's housing projection model with adjusted costs to estimate the unit difference.

## 1. Data: Predict Market Values

Fetch SF Assessor data, train a gradient boosting model on recent sales, and predict values for all residential parcels.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from pathlib import Path
import subprocess
import json
import tempfile

PROJECT_ROOT = Path.cwd().parent.parent
SOCRATA_URL = "https://data.sfgov.org/resource/wv5m-vpq2.csv"

NUMERIC_FEATURES = [
    'number_of_bathrooms', 'number_of_bedrooms', 'number_of_rooms',
    'number_of_stories', 'number_of_units', 'lot_area', 'property_area',
    'basement_area', 'lot_depth', 'lot_frontage', 'building_age',
    'percent_of_ownership', 'sqft_per_unit', 'sqft_per_bedroom',
    'has_lot_area', 'beds_imputed', 'rooms_imputed', 'baths_imputed'
]

CATEGORICAL_FEATURES = ['use_cat', 'nbhd_cat', 'zoning_cat', 'construction_cat']

RESIDENTIAL_USES = [
    'Single Family Residential',
    'Multi-Family Residential',
    'Residential Vacant Lot',
    'Residential Misc'
]

EXCLUDED_USES = [
    'Commercial Misc', 'Commercial Retail', 'Commercial Office',
    'Commercial Hotel', 'Industrial', 'Vacant Lot Comm and Ind',
    'Misc', 'Under Water Lot', 'Vacant Street Parcel', 'Vacant Lot'
]

In [2]:
print("Fetching data from SF Open Data...")
query = f"{SOCRATA_URL}?$where=closed_roll_year='2024'&$limit=300000"
df = pd.read_csv(query)
print(f"Loaded {len(df):,} parcels from 2024 tax roll")

df['assessed_land_value'] = pd.to_numeric(df['assessed_land_value'], errors='coerce').fillna(0)
df['assessed_improvement_value'] = pd.to_numeric(df['assessed_improvement_value'], errors='coerce').fillna(0)
df['total_assessed_value'] = df['assessed_land_value'] + df['assessed_improvement_value']
print(f"Calculated total_assessed_value for all parcels")

Fetching data from SF Open Data...


Loaded 212,653 parcels from 2024 tax roll
Calculated total_assessed_value for all parcels


In [3]:
def prepare_training_data(df):
    df['current_sales_date'] = pd.to_datetime(df['current_sales_date'], errors='coerce')
    df['sale_year'] = df['current_sales_date'].dt.year
    recent = df[df['current_sales_date'] > '2021-02-08'].copy()
    print(f"Recent sales (after 2021-02-08): {len(recent):,}")
    
    recent = recent[recent['sale_year'] < 2024]
    print(f"Excluding 2024+ sales: {len(recent):,}")
    
    recent = recent[recent['total_assessed_value'] > 0]
    print(f"Non-zero assessed value: {len(recent):,}")
    
    recent = recent[~recent['use_definition'].isin(EXCLUDED_USES)]
    recent = recent[recent['use_definition'].isin(RESIDENTIAL_USES) |
                    recent['use_definition'].str.contains('Residential', na=False)]
    print(f"Residential only: {len(recent):,}")
    
    return recent

train_df = prepare_training_data(df)

Recent sales (after 2021-02-08): 23,590
Excluding 2024+ sales: 19,647
Non-zero assessed value: 19,559
Residential only: 18,641


In [4]:
def build_imputation_tables(df):
    df = df.copy()
    df['sqft_bucket'] = (df['property_area'] / 250).round() * 250
    
    valid_beds = df[df['number_of_bedrooms'] > 0]
    bed_lookup = valid_beds.groupby(['use_definition', 'sqft_bucket'])['number_of_bedrooms'].median()
    bed_fallback = valid_beds.groupby('use_definition')['number_of_bedrooms'].median()
    
    valid_baths = df[df['number_of_bathrooms'] > 0]
    bath_lookup = valid_baths.groupby(['use_definition', 'sqft_bucket'])['number_of_bathrooms'].median()
    bath_fallback = valid_baths.groupby('use_definition')['number_of_bathrooms'].median()
    
    valid_rooms = df[df['number_of_rooms'] > 0]
    room_lookup = valid_rooms.groupby(['use_definition', 'sqft_bucket'])['number_of_rooms'].median()
    room_fallback = valid_rooms.groupby('use_definition')['number_of_rooms'].median()
    
    return {
        'beds': (bed_lookup, bed_fallback),
        'baths': (bath_lookup, bath_fallback),
        'rooms': (room_lookup, room_fallback)
    }

def apply_imputation(df, lookup_tables):
    df = df.copy()
    df['sqft_bucket'] = (df['property_area'] / 250).round() * 250
    
    df['beds_imputed'] = 0
    df['baths_imputed'] = 0
    df['rooms_imputed'] = 0
    
    bed_lookup, bed_fallback = lookup_tables['beds']
    bath_lookup, bath_fallback = lookup_tables['baths']
    room_lookup, room_fallback = lookup_tables['rooms']
    
    for idx in df[df['number_of_bedrooms'] == 0].index:
        use = df.loc[idx, 'use_definition']
        bucket = df.loc[idx, 'sqft_bucket']
        try:
            val = bed_lookup.loc[(use, bucket)]
        except KeyError:
            val = bed_fallback.get(use, 2)
        df.loc[idx, 'number_of_bedrooms'] = val
        df.loc[idx, 'beds_imputed'] = 1
    
    for idx in df[df['number_of_bathrooms'] == 0].index:
        use = df.loc[idx, 'use_definition']
        bucket = df.loc[idx, 'sqft_bucket']
        try:
            val = bath_lookup.loc[(use, bucket)]
        except KeyError:
            val = bath_fallback.get(use, 1)
        df.loc[idx, 'number_of_bathrooms'] = val
        df.loc[idx, 'baths_imputed'] = 1
    
    for idx in df[df['number_of_rooms'] == 0].index:
        use = df.loc[idx, 'use_definition']
        bucket = df.loc[idx, 'sqft_bucket']
        try:
            val = room_lookup.loc[(use, bucket)]
        except KeyError:
            val = room_fallback.get(use, 5)
        df.loc[idx, 'number_of_rooms'] = val
        df.loc[idx, 'rooms_imputed'] = 1
    
    return df

def engineer_features(df):
    df = df.copy()
    
    df['building_age'] = 2024 - df['year_property_built'].fillna(1950)
    df['sqft_per_unit'] = df['property_area'] / df['number_of_units'].replace(0, 1)
    df['sqft_per_bedroom'] = df['property_area'] / df['number_of_bedrooms'].replace(0, 1)
    df['has_lot_area'] = (df['lot_area'] > 0).astype(int)
    
    use_counts = df['use_definition'].value_counts()
    nbhd_counts = df['analysis_neighborhood'].value_counts()
    zoning_counts = df['zoning_code'].value_counts()
    construction_counts = df['construction_type'].value_counts()
    
    df['use_cat'] = df['use_definition'].apply(
        lambda x: x if pd.notna(x) and use_counts.get(x, 0) >= 30 else 'Other'
    )
    df['nbhd_cat'] = df['analysis_neighborhood'].apply(
        lambda x: x if pd.notna(x) and nbhd_counts.get(x, 0) >= 50 else 'Other'
    )
    df['zoning_cat'] = df['zoning_code'].apply(
        lambda x: x if pd.notna(x) and zoning_counts.get(x, 0) >= 30 else 'Other'
    )
    df['construction_cat'] = df['construction_type'].apply(
        lambda x: x if pd.notna(x) and construction_counts.get(x, 0) >= 30 else 'Other'
    )
    
    for col in NUMERIC_FEATURES:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    return df

lookup_tables = build_imputation_tables(train_df)
train_df = apply_imputation(train_df, lookup_tables)
train_df = engineer_features(train_df)
print(f"Prepared {len(train_df):,} training samples")

Prepared 18,641 training samples


In [5]:
print("Training Gradient Boosting model...")

X = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = np.log1p(train_df['total_assessed_value'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(max_categories=30, handle_unknown='ignore'), CATEGORICAL_FEATURES)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(
        n_estimators=500,
        max_depth=7,
        learning_rate=0.08,
        subsample=0.8,
        min_samples_leaf=10,
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_dollars = np.expm1(y_pred)
y_test_dollars = np.expm1(y_test)

pct_errors = np.abs(y_pred_dollars - y_test_dollars) / y_test_dollars

print(f"\nModel Performance:")
print(f"  R² (log scale): {model.score(X_test, y_test):.3f}")
print(f"  Median % error: {np.median(pct_errors)*100:.1f}%")
print(f"  Within 25%: {(pct_errors < 0.25).mean()*100:.1f}%")
print(f"  Within 50%: {(pct_errors < 0.50).mean()*100:.1f}%")

print("\nRetraining on full dataset...")
model.fit(X, y)

Training Gradient Boosting model...



Model Performance:
  R² (log scale): 0.700
  Median % error: 14.7%
  Within 25%: 73.8%
  Within 50%: 92.9%

Retraining on full dataset...


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [6]:
print("Preparing full dataset for prediction...")

all_df = df[~df['use_definition'].isin(EXCLUDED_USES)].copy()
all_df = all_df[all_df['use_definition'].isin(RESIDENTIAL_USES) |
                all_df['use_definition'].str.contains('Residential', na=False)]
print(f"Residential parcels: {len(all_df):,}")

all_df = apply_imputation(all_df, lookup_tables)
all_df = engineer_features(all_df)

all_df['current_sales_date'] = pd.to_datetime(all_df['current_sales_date'], errors='coerce')
all_df['sale_year'] = all_df['current_sales_date'].dt.year

recent_mask = (all_df['sale_year'] >= 2021) & (all_df['sale_year'] < 2024) & (all_df['total_assessed_value'] > 0)
older_mask = ~recent_mask

print(f"Recently sold (using actual): {recent_mask.sum():,}")
print(f"Older (predicting): {older_mask.sum():,}")

all_df['predicted_value'] = np.nan
all_df['expected_value'] = np.nan
all_df['value_source'] = ''

all_df.loc[recent_mask, 'predicted_value'] = all_df.loc[recent_mask, 'total_assessed_value']
all_df.loc[recent_mask, 'expected_value'] = all_df.loc[recent_mask, 'total_assessed_value']
all_df.loc[recent_mask, 'value_source'] = 'actual'

if older_mask.sum() > 0:
    X_pred = all_df.loc[older_mask, NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    log_predictions = model.predict(X_pred)
    predictions = np.expm1(log_predictions)
    all_df.loc[older_mask, 'predicted_value'] = predictions
    all_df.loc[older_mask, 'expected_value'] = predictions
    all_df.loc[older_mask, 'value_source'] = 'predicted'

print(f"\nPredicted values for {len(all_df):,} parcels")

Preparing full dataset for prediction...


Residential parcels: 191,366


Recently sold (using actual): 19,196
Older (predicting): 172,170



Predicted values for 191,366 parcels


In [7]:
print("Sample predictions:")
sample = all_df[['parcel_number', 'property_location', 'use_definition', 
                 'analysis_neighborhood', 'expected_value', 'value_source']].head(10)
sample['expected_value'] = sample['expected_value'].apply(lambda x: f"${x:,.0f}")
display(sample)

Sample predictions:


,parcel_number,property_location,use_definition,analysis_neighborhood,expected_value,value_source
0,0221071,0000 1250 JONES ST1101,Single Family Residential,Nob Hill,"$1,289,497",predicted
1,0221072,0000 1250 JONES ST1102,Single Family Residential,Nob Hill,"$1,697,932",actual
2,0221077,0000 1250 JONES ST1302,Single Family Residential,Nob Hill,"$1,974,636",predicted
3,0221079,0000 1250 JONES ST1402,Single Family Residential,Nob Hill,"$1,754,233",predicted
4,0221130,0000 1151 TAYLOR ST0002,Single Family Residential,Nob Hill,"$1,177,045",predicted
5,0221135,0000 1220 JONES ST0202,Single Family Residential,Nob Hill,"$1,137,000",actual
6,0221140,0000 1220 JONES ST0501,Single Family Residential,Nob Hill,"$3,246,884",predicted
7,0221143,0029 0025 PLEASANT ST0000,Multi-Family Residential,Nob Hill,"$1,795,794",predicted
8,0221149,0000 1224 SACRAMENTO ST0005,Single Family Residential,Nob Hill,"$716,409",predicted
9,0222022,1140 1138 TAYLOR ST0000,Multi-Family Residential,Nob Hill,"$1,650,308",predicted


## 2. Analysis: Calculate Housing Impact

Use the City Economist's housing projection model (via JS calculator) to estimate the impact of eliminating the transfer tax.

In [8]:
BASE_CONSTRUCTION_COST = 112.723
TAX_TO_COST_MULTIPLIER = 2

def get_transfer_tax_rate(value):
    if value >= 25_000_000: return 0.06
    if value >= 10_000_000: return 0.055
    if value >= 5_000_000: return 0.0225
    if value >= 1_000_000: return 0.0075
    if value >= 250_001: return 0.0068
    return 0.005

all_df['mapblklot'] = all_df['parcel_number'].str[:7]
mapblklot_values = all_df.groupby('mapblklot')['expected_value'].sum().reset_index()
mapblklot_values['tax_rate'] = mapblklot_values['expected_value'].apply(get_transfer_tax_rate)
mapblklot_values['adjusted_construction_cost'] = BASE_CONSTRUCTION_COST * (1 - mapblklot_values['tax_rate'] * TAX_TO_COST_MULTIPLIER)

print(f"Aggregated {len(all_df):,} parcels into {len(mapblklot_values):,} mapblklots")
print(f"\nTax bracket distribution:")
display(mapblklot_values['tax_rate'].value_counts().sort_index())

Aggregated 191,366 parcels into 162,862 mapblklots

Tax bracket distribution:


tax_rate
0.0050       182
0.0068     30050
0.0075    124060
0.0225      5470
0.0550      2565
0.0600       535
Name: count, dtype: int64

In [9]:
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
    mapblklot_values[['mapblklot', 'adjusted_construction_cost']].to_csv(f, index=False)
    costs_csv_path = f.name

print("Running housing projection model...")

result = subprocess.run(
    ['npx', 'vite-node', 'analyses/transfer-tax-reform/calculate-expected-units.mjs', costs_csv_path],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)

import os
os.unlink(costs_csv_path)

if result.returncode != 0:
    print("Error running calculation:")
    print("STDERR:", result.stderr)
    print("STDOUT:", result.stdout)
else:
    results = json.loads(result.stdout)
    print("Calculation complete!")

Running housing projection model...


Calculation complete!


In [10]:
results_df = pd.DataFrame({
    'Scenario': ['Low Growth', 'High Growth'],
    'Original (no reform)': [f"{results['original']['low']:,}", f"{results['original']['high']:,}"],
    'With Reform': [f"{results['withReform']['low']:,}", f"{results['withReform']['high']:,}"],
    'Difference': [
        f"+{results['difference']['low']:,} (+{results['difference']['lowPct']}%)",
        f"+{results['difference']['high']:,} (+{results['difference']['highPct']}%)"
    ]
})

display(results_df)

,Scenario,Original (no reform),With Reform,Difference
0,Low Growth,"44,918","50,776","+5,858 (+13.0%)"
1,High Growth,"76,411","87,336","+10,925 (+14.3%)"


## 3. Results Summary

### Key Finding

Eliminating San Francisco's transfer tax would result in an estimated **+5,858 to +10,925 additional housing units** over 20 years, representing a **13-14% increase** over the baseline projection.

### Methodology Notes

- **Transfer tax impact**: Land value is ~2x construction costs in SF. Eliminating the transfer tax reduces land acquisition costs, with a 2x effect on effective construction cost index.
- **Model**: Uses the City Economist's probability/units model for housing projection.
- **Tax brackets**: SF transfer tax rates range from 0.5% to 6% based on property value.

### SF Transfer Tax Brackets

| Property Value | Tax Rate |
|----------------|----------|
| $100 - $250,000 | 0.50% |
| $250,001 - $999,999 | 0.68% |
| $1,000,000 - $4,999,999 | 0.75% |
| $5,000,000 - $9,999,999 | 2.25% |
| $10,000,000 - $24,999,999 | 5.50% |
| $25,000,000+ | 6.00% |

### Data Sources

- SF Assessor Historical Secured Property Tax Rolls (via Socrata API)
- City Economist housing projection model

---
*Analysis performed: March 2025*